# Divide and conquer Delaunay triangulation

Runs [divide_and_conquer.py](divide_and_conquer.py) on four point sets. In each picture, the left side shows
the centre of mass split lines by recursion depth and the leaves (points, segments, triangles). The middle
shows the merged Delaunay triangulation with the top split line and its two bridges. The right side shows its
dual, the Voronoi diagram: circumcentres joined across every inner edge, and a short dotted stump outwards
past every hull edge, standing in for its ray to infinity.

In [ ]:
import numpy as np

from divide_and_conquer import Triangulation, draw

SEED = 1

def triangulate_and_draw(points, title):
    t = Triangulation(points)
    print(f'{len(points)} points: {len(t.triangles())} triangles, hull of {len(t.hull)}, '
          f'{t.flips} flips, {len(t.illegal_edges())} illegal edges left')
    draw(t, title)

## 27 points, uniform in the unit square

The same points as `python divide_and_conquer.py` with its default seed.

In [ ]:
triangulate_and_draw(np.random.default_rng(SEED).uniform(size=(27, 2)),
                     f'27 uniform random points (seed {SEED})')

## 225 points, uniform in the unit square

In [ ]:
triangulate_and_draw(np.random.default_rng(SEED).uniform(size=(225, 2)),
                     f'225 uniform random points (seed {SEED})')

## A donut: points uniform between two circles

Uniform by area: the radius is the square root of a uniform pick between the squared radii, or points would
crowd the inner circle. The top merge zips across the hole.

In [ ]:
def donut(n, inner, outer, seed):
    rng = np.random.default_rng(seed)
    radius = np.sqrt(rng.uniform(inner ** 2, outer ** 2, n))
    angle = rng.uniform(0, 2 * np.pi, n)
    return np.c_[radius * np.cos(angle), radius * np.sin(angle)]

triangulate_and_draw(donut(300, inner=0.5, outer=1.0, seed=SEED),
                     f'300 random points between circles of radius 0.5 and 1 (seed {SEED})')

## A circle and a semicircle

100 points evenly on a circle, and 100 on a smaller semicircle with a flat bottom, shifted up and to the right.
The joined convex hull runs along a straight tangent between the two round curves at the top.

All the points of each shape lie on one circle, so their Delaunay triangulation is not unique: any triangulation
of a shape's points is Delaunay. The script's circle test has a tolerance for that, or rounding noise would
flip such edges back and forth.

In [ ]:
def circle_and_semicircle(n=100):
    angle = 2 * np.pi * np.arange(n) / n
    circle = np.c_[np.cos(angle), np.sin(angle)]
    # Both ends of the semicircle included: they are the corners of its flat bottom
    angle = np.pi * np.arange(n) / (n - 1)
    semicircle = np.c_[2.2 + 0.8 * np.cos(angle), 0.9 + 0.8 * np.sin(angle)]
    return np.r_[circle, semicircle]

triangulate_and_draw(circle_and_semicircle(), '100 points on a circle and 100 on a semicircle')